<div class='alert alert-warning'>

Интерактивные примеры NumPy — экспериментальная возможность, поэтому они не всегда работают как ожидается и могут долго загружаться, особенно на устройствах с ограниченными ресурсами. Версия NumPy в примерах также может отличаться от версии, для которой вы просматриваете документацию. Если возникнут проблемы, сообщите о них в [систему отслеживания ошибок NumPy](https://github.com/numpy/numpy/issues).

</div>

In [ ]:
import numpy as np
from io import StringIO   # StringIO behaves like a file object
c = StringIO("0 1\n2 3")
np.loadtxt(c)

array([[0., 1.],
       [2., 3.]])

**Комментарии к примеру на русском**

StringIO ведёт себя как файловый объект

In [ ]:
d = StringIO("M 21 72\nF 35 58")
np.loadtxt(d, dtype={'names': ('gender', 'age', 'weight'),
                     'formats': ('S1', 'i4', 'f4')})

array([(b'M', 21, 72.), (b'F', 35, 58.)],
      dtype=[('gender', 'S1'), ('age', '<i4'), ('weight', '<f4')])

In [ ]:
c = StringIO("1,0,2\n3,0,4")
x, y = np.loadtxt(c, delimiter=',', usecols=(0, 2), unpack=True)
x

array([1., 3.])

In [ ]:
y

array([2., 4.])

Аргумент `converters` задаёт функции предварительной обработки текста перед его разбором. `converters` может быть словарём, сопоставляющим каждому столбцу функцию предварительной обработки:


In [ ]:
s = StringIO("1.618, 2.296\n3.141, 4.669\n")
conv = {
    0: lambda x: np.floor(float(x)),  # conversion fn for column 0
    1: lambda x: np.ceil(float(x)),  # conversion fn for column 1
}
np.loadtxt(s, delimiter=",", converters=conv)

array([[1., 3.],
       [3., 5.]])

**Комментарии к примеру на русском**

Функция преобразования столбца 0

Функция преобразования столбца 1

`converters` может быть вызываемым объектом вместо словаря; тогда он применяется ко всем столбцам:


In [ ]:
s = StringIO("0xDE 0xAD\n0xC0 0xDE")
import functools
conv = functools.partial(int, base=16)
np.loadtxt(s, converters=conv)

array([[222., 173.],
       [192., 222.]])

Этот пример показывает, как с помощью `converters` преобразовать поле со знаком минуса в конце в отрицательное число.


In [ ]:
s = StringIO("10.01 31.25-\n19.22 64.31\n17.57- 63.94")
def conv(fld):
    return -float(fld[:-1]) if fld.endswith("-") else float(fld)

np.loadtxt(s, converters=conv)

array([[ 10.01, -31.25],
       [ 19.22,  64.31],
       [-17.57,  63.94]])

Вызываемый объект в качестве преобразователя особенно полезен при обработке
значений в разных форматах, например чисел с плавающей точкой с символами подчёркивания:


In [ ]:
s = StringIO("1 2.7 100_000")
np.loadtxt(s, converters=float)

array([1.e+00, 2.7e+00, 1.e+05])

Эта идея может быть расширена для автоматической обработки значений, заданных во многих различных форматах, таких как шестнадцатеричные значения:

In [ ]:
def conv(val):
    try:
        return float(val)
    except ValueError:
        return float.fromhex(val)
s = StringIO("1, 2.5, 3_000, 0b4, 0x1.4000000000000p+2")
np.loadtxt(s, delimiter=",", converters=conv)

array([1.0e+00, 2.5e+00, 3.0e+03, 1.8e+02, 5.0e+00])

Или формат, в котором знак ``-`` стоит после числа:


In [ ]:
s = StringIO("10.01 31.25-\n19.22 64.31\n17.57- 63.94")
conv = lambda x: -float(x[:-1]) if x.endswith("-") else float(x)
np.loadtxt(s, converters=conv)

array([[ 10.01, -31.25],
       [ 19.22,  64.31],
       [-17.57,  63.94]])

Поддержка полей в кавычках включается параметром `quotechar`. Символы комментария и разделителя игнорируются внутри поля, ограниченного `quotechar`:

In [ ]:
s = StringIO('"alpha, #42", 10.0\n"beta, #64", 2.0\n')
dtype = np.dtype([("label", "U12"), ("value", float)])
np.loadtxt(s, dtype=dtype, delimiter=",", quotechar='"')

array([('alpha, #42', 10.), ('beta, #64',  2.)],
      dtype=[('label', '<U12'), ('value', '<f8')])

Поля в кавычках могут разделяться несколькими пробельными символами:


In [ ]:
s = StringIO('"alpha, #42"       10.0\n"beta, #64" 2.0\n')
dtype = np.dtype([("label", "U12"), ("value", float)])
np.loadtxt(s, dtype=dtype, delimiter=None, quotechar='"')

array([('alpha, #42', 10.), ('beta, #64',  2.)],
      dtype=[('label', '<U12'), ('value', '<f8')])

Две подряд идущие кавычки внутри поля, заключённого в кавычки, интерпретируются как одна экранированная кавычка:


In [ ]:
s = StringIO('"Hello, my name is ""Monty""!"')
np.loadtxt(s, dtype=np.str_, delimiter=",", quotechar='"')

array('Hello, my name is "Monty"!', dtype='<U26')

Чтение части столбцов, когда строки содержат разное число значений:


In [ ]:
d = StringIO("1 2\n2 4\n3 9 12\n4 16 20")
np.loadtxt(d, usecols=(0, 1))

array([[ 1.,  2.],
       [ 2.,  4.],
       [ 3.,  9.],
       [ 4., 16.]])